<a href="https://colab.research.google.com/github/adithya70/Streamlit-with-Retrieval-Augmented-Generation-RAG-Using-Project-Datasets/blob/main/Streamlit_with_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit sentence-transformers faiss-cpu pyngrok pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20

In [ ]:
!mkdir ~/.kaggle
!mv kaggle.json ~/.kaggle/

In [ ]:
!kaggle datasets download -d infamouscoder/depression-reddit-cleaned

Dataset URL: https://www.kaggle.com/datasets/infamouscoder/depression-reddit-cleaned
License(s): CC0-1.0
  0% 0.00/979k [00:00<?, ?B/s]
100% 979k/979k [00:00<00:00, 148MB/s]


In [ ]:
!unzip depression-reddit-cleaned.zip -d data/

Archive:  depression-reddit-cleaned.zip
  inflating: data/depression_dataset_reddit_cleaned.csv  


In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("/content/data/depression_dataset_reddit_cleaned.csv")
print(df.head())

                                          clean_text  is_depression
0  we understand that most people who reply immed...              1
1  welcome to r depression s check in post a plac...              1
2  anyone else instead of sleeping more when depr...              1
3  i ve kind of stuffed around a lot in my life d...              1
4  sleep is my greatest and most comforting escap...              1


In [ ]:
df = df[df["is_depression"] == 1]


In [ ]:
df.to_csv("data/processed_dataset.csv", index=False)

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the model
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:


# Generate embeddings
embeddings = model.encode(df["clean_text"].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/120 [00:00<?, ?it/s]

In [ ]:
df.head()

,clean_text,is_depression
0,we understand that most people who reply immed...,1
1,welcome to r depression s check in post a plac...,1
2,anyone else instead of sleeping more when depr...,1
3,i ve kind of stuffed around a lot in my life d...,1
4,sleep is my greatest and most comforting escap...,1


In [ ]:
import faiss

# Create a FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)  # L2 distance for similarity search
index.add(embeddings)  # Add embeddings to the index

In [ ]:
def retrieve(query, index, df, model, top_k=5):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, top_k)
    results = df.iloc[indices[0]]
    return results

In [ ]:

from pyngrok import ngrok

# Set your ngrok auth token
ngrok.set_auth_token("2FmFzkDXLyfLTlWyu2zlfi8ovSw_2t74v3BtP2DUCd6uwD7Bk")

In [ ]:
%%writefile app.py
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow logs
import warnings
warnings.filterwarnings("ignore", category=UserWarning)  # Suppress PyTorch warnings

import streamlit as st
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load the dataset
@st.cache_data
def load_data():
    df = pd.read_csv("/content/data/processed_dataset.csv")
    return df

# Load the Sentence Transformer model
@st.cache_resource
def load_model():
    return SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings and build FAISS index
@st.cache_resource
def build_faiss_index(_model, df):
    embeddings = _model.encode(df["clean_text"].tolist(), show_progress_bar=True)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return index

# Retrieve relevant results
def retrieve(query, index, df, model, top_k=5):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, top_k)
    results = df.iloc[indices[0]]
    return results

# Streamlit app
def main():
    st.title("Depression Reddit Post Retrieval")

    # Load data and model
    df = load_data()
    model = load_model()
    index = build_faiss_index(_model=model, df=df)  # Pass _model instead of model

    # Input box for user query
    query = st.text_input("Enter your query:")

    # Search button
    if st.button("Search"):
        if query:
            results = retrieve(query, index, df, model)
            st.write("### Search Results:")
            for i, row in results.iterrows():
                st.markdown(f"""
                **Result {i+1}:**
                - **Text:** {row['clean_text']}
                - **Depression Label:** {"Depression" if row['is_depression'] == 1 else "Not Depression"}
                """)
        else:
            st.warning("Please enter a query.")

if __name__ == "__main__":
    main()

Overwriting app.py


In [ ]:
!streamlit run app.py & ngrok http 8501





  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.134.235:8501

E0000 00:00:1739319614.211758   19008 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739319614.219895   19008 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Batches: 100% 120/120 [00:06<00:00, 19.15it/s]
2025-02-12 00:20:25.844 Examining the path of torch.classes raised:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/streamlit/watcher/local_sources_watcher.py", line 217, in get_module_paths
    potential_paths = extract_paths(module)
                      ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/streamlit/watcher/local_sources_watcher.py", line 210, in <la